In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [5]:
df_songs = pd.read_csv('dataset.csv')

In [6]:
# Split the data into features (X) and target (y)
X = df_songs['lyrics_cleaned']
y = df_songs['parent_genre']

# Split into training and test sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create a pipeline: Vectorizer -> Logistic Regression
# We use 'lbfgs' solver which is good for multiclass problems
model_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english')),
    ('clf', LogisticRegression(solver='lbfgs', max_iter=1000,class_weight='balanced'))
])

# Train the model
print("Training model...")
model_pipeline.fit(X_train, y_train)

# Make predictions
predictions = model_pipeline.predict(X_test)

# Evaluate
print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

Training model...
Accuracy: 0.38

Classification Report:

                  precision    recall  f1-score   support

       Asian Pop       0.38      0.46      0.42        96
       Classical       0.10      0.34      0.15        44
      Electronic       0.56      0.27      0.37       882
    Folk/Country       0.45      0.55      0.49       322
   Hip-Hop & R&B       0.21      0.59      0.31        29
    Jazz & Blues       0.13      0.33      0.19        57
           Latin       0.06      0.33      0.11        12
           Metal       0.54      0.71      0.61       452
             Pop       0.21      0.25      0.23       285
Reggae/Caribbean       0.43      0.51      0.47        86
            Rock       0.28      0.14      0.18       493
       Soul/Funk       0.17      0.27      0.21       157
  World/Regional       0.67      0.67      0.67       172

        accuracy                           0.38      3087
       macro avg       0.32      0.42      0.34      3087
    weighted

In [11]:
y_prob = pd.DataFrame(model_pipeline.predict_proba(X_test))
classes = model_pipeline.classes_
class_list = []
for item in class_list:
    class_list.append(str(item))
indicies = range(0,len(classes))
map = {}
for i in indicies:
    map[i] = classes[i]

y_prob.rename(columns=map)

y_prob['Top_3_Cols'] = y_prob.apply(lambda row: row.nlargest(3).index.map(map).tolist(), axis=1)

top_3 = y_prob['Top_3_Cols'].to_list()

In [12]:
def in_top(pred_list,actual, top_x):
    return True if actual in pred_list[:top_x] else False

In [14]:
predictions = pd.DataFrame({
    'id': X_test.index,
    'predicted': top_3,
    'actual' : y_test
})

for x in range(1,4):
    predictions[f'In Top {x}'] = predictions.apply(lambda row: in_top(row['predicted'],row['actual'],x), axis=1)


